# 1. Project Initialization for Target Construction

### Objective
This section initializes the environment for constructing and validating the six-month forward-looking credit-risk early-warning target.

### Why this is necessary
The target variable is the foundation of the subsequent feature engineering and modelling stages. Therefore, the target must be constructed from the validated source data using an explicit and reproducible methodology.

The target is defined at loan-month level as follows:

- The observation at month `t` must have a current delinquency status of `00`.
- A positive target (`1`) is assigned if the loan becomes 30 or more days delinquent at any point during the next six actual calendar months (`t+1` through `t+6`).
- A negative target (`0`) is assigned if the loan remains current (`00`) throughout all six future months.
- The target is excluded when:
  - any of the six required future calendar months is unavailable; or
  - any required future delinquency status is `XX`.

### Important methodological principle
Future observations are matched using actual calendar months rather than row positions. This prevents reporting gaps from incorrectly being interpreted as consecutive monthly observations.

The source dataset is read from the validated processed chunks using DuckDB so that the full dataset does not need to be loaded into pandas.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import duckdb
import glob
import gc

# -------------------------------------------------------------------
# Project paths
# -------------------------------------------------------------------

PROJECT_DIR = Path(
    r"C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning"
)

DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
CHUNKS_DIR = PROCESSED_DIR / "fannie_2018Q1_chunks"

TARGET_DIR = PROCESSED_DIR
TARGET_FILE = TARGET_DIR / "fannie_2018Q1_early_warning_target_corrected.parquet"

RAW_GLOB = str(CHUNKS_DIR / "part_*.csv.gz")

FILES = sorted(glob.glob(RAW_GLOB))

# -------------------------------------------------------------------
# DuckDB connection
# -------------------------------------------------------------------

con = duckdb.connect()

con.execute("SET threads = 2")
con.execute("SET preserve_insertion_order = false")

# -------------------------------------------------------------------
# Basic environment validation
# -------------------------------------------------------------------

print("=" * 70)
print("TARGET CONSTRUCTION ENVIRONMENT")
print("=" * 70)

print(f"Project directory : {PROJECT_DIR}")
print(f"Chunks directory  : {CHUNKS_DIR}")
print(f"Number of chunks  : {len(FILES)}")
print(f"DuckDB version    : {duckdb.__version__}")

print("\nPath checks:")
print(f"Project exists    : {PROJECT_DIR.exists()}")
print(f"Chunks exists     : {CHUNKS_DIR.exists()}")

if len(FILES) == 0:
    raise FileNotFoundError(
        "No processed Fannie Mae data chunks were found."
    )

print("\nFirst chunk:")
print(FILES[0])

print("\nLast chunk:")
print(FILES[-1])

print("\nTarget definition:")
print("Eligible observation : current status = 00")
print("Forecast horizon     : next 6 calendar months")
print("Event                : any future status >= 01")
print("Unknown future       : XX -> exclude")
print("Missing future month : exclude")

TARGET CONSTRUCTION ENVIRONMENT
Project directory : C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning
Chunks directory  : C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\fannie_2018Q1_chunks
Number of chunks  : 95
DuckDB version    : 1.5.5

Path checks:
Project exists    : True
Chunks exists     : True

First chunk:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\fannie_2018Q1_chunks\part_0001.csv.gz

Last chunk:
C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\fannie_2018Q1_chunks\part_0095.csv.gz

Target definition:
Eligible observation : current status = 00
Forecast horizon     : next 6 calendar months
Event                : any future status >= 01
Unknown future       : XX -> exclude
Missing future month : exclude


# 2. Build the Valid Loan-Month Panel

### Objective
This section creates the loan-month panel required for six-month forward target construction.

### Why this is necessary
The source dataset contains monthly observations for individual mortgage loans. To determine whether a loan becomes delinquent during the next six calendar months, observations must first be represented using an explicit calendar date.

The panel contains only the variables required for target construction:

- `loan_id`
- `monthly_reporting_period`
- parsed `reporting_date`
- `current_delinquency_status`

### Methodological principle
The reporting period is converted from the source character format (`MMYYYY`) into a true calendar date.

The target construction will subsequently match each observation at month `t` with the exact calendar months:

`t+1, t+2, t+3, t+4, t+5, t+6`

This approach avoids incorrectly treating a later observation as a consecutive month when an intermediate reporting month is missing.

### Important note
Only observations with current delinquency status `00` are eligible for the early-warning target. However, all delinquency observations are retained in this panel because they are required to determine future outcomes.

No target is assigned in this step and no source observations are modified.

In [2]:
# Create the loan-month panel required for target construction.
# The result is materialized as a temporary DuckDB table so that
# subsequent target calculations can reuse it without rereading
# the compressed CSV files.

con.execute("DROP TABLE IF EXISTS loan_month_panel")

panel_query = f"""
CREATE TEMP TABLE loan_month_panel AS
SELECT
    loan_id,
    monthly_reporting_period,
    STRPTIME(
        monthly_reporting_period,
        '%m%Y'
    ) AS reporting_date,
    current_delinquency_status
FROM read_csv_auto(
    '{RAW_GLOB}',
    header = true,
    compression = 'gzip',
    union_by_name = true
)
"""

con.execute(panel_query)

panel_summary = con.execute("""
SELECT
    COUNT(*) AS total_observations,
    COUNT(DISTINCT loan_id) AS unique_loans,
    COUNT(DISTINCT reporting_date) AS reporting_months,
    MIN(reporting_date) AS earliest_reporting_date,
    MAX(reporting_date) AS latest_reporting_date,
    COUNT(*) FILTER (
        WHERE current_delinquency_status = '00'
    ) AS current_observations
FROM loan_month_panel
""").fetchdf()

print("=" * 70)
print("LOAN-MONTH PANEL")
print("=" * 70)

display(panel_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

LOAN-MONTH PANEL


,total_observations,unique_loans,reporting_months,earliest_reporting_date,latest_reporting_date,current_observations
0,23563399,458830,99,2018-01-01,2026-03-01,22823077


# 3. Construct the Six-Month Forward Early-Warning Target

### Objective
This section constructs the six-month forward-looking early-warning target for currently performing mortgage loans.

### Target definition

For an observation at month `t`:

- The observation must have current delinquency status `00`.
- Target `1` is assigned if the loan becomes 30 or more days delinquent in any of the next six calendar months.
- Target `0` is assigned if the loan remains current throughout all six future calendar months.
- The observation is excluded if any required future month is missing or any future status is `XX`.

### Calendar continuity

The six-month horizon is based on actual calendar months rather than row positions.

For each loan, the sixth future observation must occur exactly six calendar months after the observation month. Because the loan-month uniqueness validation has already confirmed that each loan-month combination is unique, this guarantees that the required six consecutive monthly observations exist.

### Memory management

The dataset contains more than 23 million observations. Therefore, the target is constructed using DuckDB window functions with a dedicated disk-backed database and temporary directory.

A fresh DuckDB connection is created before any query is executed so that memory and temporary-storage settings can be configured correctly.

No source observations are modified.

In [5]:
# ================================================================
# MEMORY-SAFE TARGET CONSTRUCTION
# ================================================================

# Close the previous DuckDB connection so that memory from the
# failed target-construction attempt can be released.
try:
    con.close()
except:
    pass

gc.collect()

# ----------------------------------------------------------------
# Create dedicated directories for target construction
# ----------------------------------------------------------------

TARGET_DB_DIR = PROCESSED_DIR / "target_construction_db"
TARGET_TEMP_DIR = PROCESSED_DIR / "target_construction_temp"

TARGET_DB_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TARGET_TEMP_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TARGET_DB_FILE = TARGET_DB_DIR / "target_construction.duckdb"

# ----------------------------------------------------------------
# Fresh DuckDB connection
# ----------------------------------------------------------------

target_con = duckdb.connect(
    str(TARGET_DB_FILE)
)

# IMPORTANT:
# These settings are applied BEFORE any query is executed.
target_con.execute(
    "SET memory_limit = '4GB'"
)

target_con.execute(
    "SET threads = 2"
)

target_con.execute(
    "SET preserve_insertion_order = false"
)

target_con.execute(
    f"SET temp_directory = '{TARGET_TEMP_DIR}'"
)

# ----------------------------------------------------------------
# Construct target directly from the source chunks.
# Only the four required fields are read.
# ----------------------------------------------------------------

target_con.execute(
    "DROP TABLE IF EXISTS early_warning_target"
)

target_query = f"""
CREATE TABLE early_warning_target AS

WITH ordered AS (

    SELECT
        loan_id,
        reporting_date,
        current_delinquency_status,

        LEAD(reporting_date, 6) OVER (
            PARTITION BY loan_id
            ORDER BY reporting_date
        ) AS date_t6,

        MAX(
            CASE
                WHEN TRY_CAST(
                    current_delinquency_status AS INTEGER
                ) >= 1
                THEN 1
                ELSE 0
            END
        ) OVER (
            PARTITION BY loan_id
            ORDER BY reporting_date
            ROWS BETWEEN 1 FOLLOWING AND 6 FOLLOWING
        ) AS future_delinquency,

        MAX(
            CASE
                WHEN current_delinquency_status = 'XX'
                THEN 1
                ELSE 0
            END
        ) OVER (
            PARTITION BY loan_id
            ORDER BY reporting_date
            ROWS BETWEEN 1 FOLLOWING AND 6 FOLLOWING
        ) AS future_unknown_status

    FROM (
        SELECT
            loan_id,

            STRPTIME(
                monthly_reporting_period,
                '%m%Y'
            ) AS reporting_date,

            current_delinquency_status

        FROM read_csv_auto(
            '{RAW_GLOB}',
            header = true,
            compression = 'gzip',
            union_by_name = true
        )
    ) source_data
)

SELECT
    loan_id,
    reporting_date,

    CASE
        WHEN date_t6 IS NULL
            THEN NULL

        WHEN date_t6 <>
             reporting_date + INTERVAL '6 months'
            THEN NULL

        WHEN future_unknown_status = 1
            THEN NULL

        WHEN future_delinquency = 1
            THEN 1

        ELSE 0
    END AS target_6m,

    CASE
        WHEN date_t6 IS NULL
            THEN 'MISSING_FUTURE_MONTH'

        WHEN date_t6 <>
             reporting_date + INTERVAL '6 months'
            THEN 'MISSING_FUTURE_MONTH'

        WHEN future_unknown_status = 1
            THEN 'UNKNOWN_FUTURE_STATUS'

        WHEN future_delinquency = 1
            THEN 'VALID_EVENT'

        ELSE 'VALID_NON_EVENT'
    END AS target_status

FROM ordered

WHERE current_delinquency_status = '00'
"""

print("=" * 70)
print("CONSTRUCTING SIX-MONTH EARLY-WARNING TARGET")
print("=" * 70)
print("Fresh DuckDB connection created.")
print("Memory limit     : 4 GB")
print("Threads          : 2")
print(f"Temporary folder : {TARGET_TEMP_DIR}")
print()
print("Processing 23+ million loan-month observations...")
print("Disk spilling is enabled for large intermediate operations.")
print()

target_con.execute(target_query)

print("=" * 70)
print("TARGET CONSTRUCTION COMPLETED")
print("=" * 70)

CONSTRUCTING SIX-MONTH EARLY-WARNING TARGET
Fresh DuckDB connection created.
Memory limit     : 4 GB
Threads          : 2
Temporary folder : C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\target_construction_temp

Processing 23+ million loan-month observations...
Disk spilling is enabled for large intermediate operations.



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

TARGET CONSTRUCTION COMPLETED


# 4. Target Distribution and Validity Validation

### Objective
This section validates the distribution and temporal coverage of the newly constructed six-month early-warning target.

### Why this is necessary
The target variable must be validated before it is used for feature engineering or model development.

The following quantities are examined:

- total eligible current loan-month observations;
- valid event observations (`target = 1`);
- valid non-event observations (`target = 0`);
- excluded observations;
- overall event rate;
- target-status categories;
- earliest and latest observation dates;
- earliest and latest dates for valid targets.

### Expected temporal result
Because the source dataset ends in March 2026, an observation requires six complete future calendar months to receive a valid target.

Therefore, the latest possible valid observation month is **September 2025**.

This provides an important independent check that the calendar-based target construction has been implemented correctly.

No observations are modified in this validation step.

In [6]:
# ================================================================
# TARGET DISTRIBUTION AND VALIDITY VALIDATION
# ================================================================

target_validation_query = """
SELECT
    COUNT(*) AS eligible_observations,

    COUNT(*) FILTER (
        WHERE target_6m = 1
    ) AS target_events,

    COUNT(*) FILTER (
        WHERE target_6m = 0
    ) AS target_non_events,

    COUNT(*) FILTER (
        WHERE target_6m IS NULL
    ) AS excluded_observations,

    COUNT(*) FILTER (
        WHERE target_status = 'VALID_EVENT'
    ) AS valid_event_status,

    COUNT(*) FILTER (
        WHERE target_status = 'VALID_NON_EVENT'
    ) AS valid_non_event_status,

    COUNT(*) FILTER (
        WHERE target_status = 'MISSING_FUTURE_MONTH'
    ) AS missing_future_month,

    COUNT(*) FILTER (
        WHERE target_status = 'UNKNOWN_FUTURE_STATUS'
    ) AS unknown_future_status,

    MIN(reporting_date) AS earliest_observation_date,

    MAX(reporting_date) AS latest_observation_date,

    MIN(reporting_date) FILTER (
        WHERE target_6m IS NOT NULL
    ) AS earliest_valid_target_date,

    MAX(reporting_date) FILTER (
        WHERE target_6m IS NOT NULL
    ) AS latest_valid_target_date

FROM early_warning_target
"""

target_validation_df = target_con.execute(
    target_validation_query
).fetchdf()

# ---------------------------------------------------------------
# Calculate event rate
# ---------------------------------------------------------------

events = int(
    target_validation_df.loc[0, "target_events"]
)

non_events = int(
    target_validation_df.loc[0, "target_non_events"]
)

valid_targets = events + non_events

event_rate = (
    events / valid_targets * 100
    if valid_targets > 0
    else np.nan
)

target_validation_df["event_rate_percent"] = event_rate

print("=" * 70)
print("SIX-MONTH TARGET VALIDATION")
print("=" * 70)

display(target_validation_df)

print("\nTarget-status distribution:")

status_distribution = target_con.execute("""
SELECT
    target_status,
    COUNT(*) AS observations
FROM early_warning_target
GROUP BY target_status
ORDER BY observations DESC
""").fetchdf()

status_distribution["percentage"] = (
    status_distribution["observations"]
    / len(target_con.execute(
        "SELECT * FROM early_warning_target"
    ).fetchdf())
    * 100
)

display(status_distribution)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SIX-MONTH TARGET VALIDATION


,eligible_observations,target_events,target_non_events,excluded_observations,valid_event_status,valid_non_event_status,missing_future_month,unknown_future_status,earliest_observation_date,latest_observation_date,earliest_valid_target_date,latest_valid_target_date,event_rate_percent
0,22823077,646957,19508312,2667808,646957,19508312,2583082,84726,2018-01-01,2026-03-01,2018-01-01,2025-09-01,3.209865



Target-status distribution:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,target_status,observations,percentage
0,VALID_NON_EVENT,19508312,85.476257
1,MISSING_FUTURE_MONTH,2583082,11.317852
2,VALID_EVENT,646957,2.834662
3,UNKNOWN_FUTURE_STATUS,84726,0.371230


# 6. Direct Audit of Target Logic

### Objective
This section performs a direct row-level audit of the generated six-month early-warning target.

### Why this is necessary
Aggregate target counts confirm the overall structure of the target, but a direct audit is required to verify that the target values correspond to the actual future delinquency statuses.

A sample of valid event and non-event observations is selected from the constructed target table.

For each sampled observation, the actual delinquency status is then retrieved for:

- `t+1`
- `t+2`
- `t+3`
- `t+4`
- `t+5`
- `t+6`

### Validation rule

For `target_6m = 1`:
- at least one of the six future statuses must be `01` or greater.

For `target_6m = 0`:
- all six future statuses must be `00`.

This provides a direct verification that the implemented target logic agrees with the formal research definition.

No observations are modified.

In [8]:
# ================================================================
# DIRECT TARGET LOGIC AUDIT
# ================================================================

# Select a small sample of valid events and non-events.
# Only the selected loan IDs are subsequently examined.

sample_targets = target_con.execute("""
WITH events AS (
    SELECT
        loan_id,
        reporting_date,
        target_6m,
        target_status
    FROM early_warning_target
    WHERE target_6m = 1
    USING SAMPLE 10 ROWS
),

non_events AS (
    SELECT
        loan_id,
        reporting_date,
        target_6m,
        target_status
    FROM early_warning_target
    WHERE target_6m = 0
    USING SAMPLE 10 ROWS
)

SELECT *
FROM events

UNION ALL

SELECT *
FROM non_events
""").fetchdf()

print("=" * 70)
print("SELECTED AUDIT SAMPLE")
print("=" * 70)

display(sample_targets)

# ---------------------------------------------------------------
# Build a small SQL VALUES clause from the selected sample.
# This avoids depending on the old temporary loan_month_panel.
# ---------------------------------------------------------------

sample_values = []

for _, row in sample_targets.iterrows():

    sample_values.append(
        f"""
        (
            {int(row["loan_id"])},
            DATE '{pd.Timestamp(row["reporting_date"]).strftime("%Y-%m-%d")}',
            {int(row["target_6m"])},
            '{row["target_status"]}'
        )
        """
    )

sample_values_sql = ",\n".join(sample_values)

# ---------------------------------------------------------------
# Retrieve the six actual future statuses from the source data.
# ---------------------------------------------------------------

audit_query = f"""
WITH sample AS (

    SELECT *
    FROM (
        VALUES
        {sample_values_sql}
    ) AS v(
        loan_id,
        reporting_date,
        target_6m,
        target_status
    )

),

source_data AS (

    SELECT
        loan_id,

        STRPTIME(
            monthly_reporting_period,
            '%m%Y'
        ) AS reporting_date,

        current_delinquency_status

    FROM read_csv_auto(
        '{RAW_GLOB}',
        header = true,
        compression = 'gzip',
        union_by_name = true
    )

)

SELECT
    s.loan_id,
    s.reporting_date,
    s.target_6m,
    s.target_status,

    f1.current_delinquency_status AS status_t1,
    f2.current_delinquency_status AS status_t2,
    f3.current_delinquency_status AS status_t3,
    f4.current_delinquency_status AS status_t4,
    f5.current_delinquency_status AS status_t5,
    f6.current_delinquency_status AS status_t6

FROM sample s

LEFT JOIN source_data f1
    ON s.loan_id = f1.loan_id
   AND f1.reporting_date =
       s.reporting_date + INTERVAL '1 month'

LEFT JOIN source_data f2
    ON s.loan_id = f2.loan_id
   AND f2.reporting_date =
       s.reporting_date + INTERVAL '2 months'

LEFT JOIN source_data f3
    ON s.loan_id = f3.loan_id
   AND f3.reporting_date =
       s.reporting_date + INTERVAL '3 months'

LEFT JOIN source_data f4
    ON s.loan_id = f4.loan_id
   AND f4.reporting_date =
       s.reporting_date + INTERVAL '4 months'

LEFT JOIN source_data f5
    ON s.loan_id = f5.loan_id
   AND f5.reporting_date =
       s.reporting_date + INTERVAL '5 months'

LEFT JOIN source_data f6
    ON s.loan_id = f6.loan_id
   AND f6.reporting_date =
       s.reporting_date + INTERVAL '6 months'

ORDER BY
    target_6m DESC,
    reporting_date,
    loan_id
"""

target_audit_df = target_con.execute(
    audit_query
).fetchdf()

print("=" * 70)
print("DIRECT TARGET LOGIC AUDIT")
print("=" * 70)

display(target_audit_df)

SELECTED AUDIT SAMPLE


,loan_id,reporting_date,target_6m,target_status
0,171644560152,2022-08-01,0,VALID_NON_EVENT
1,215766289994,2018-08-01,0,VALID_NON_EVENT
2,227251848219,2018-07-01,0,VALID_NON_EVENT
3,252500752450,2021-11-01,0,VALID_NON_EVENT
4,269907063775,2018-04-01,0,VALID_NON_EVENT
5,450649322097,2025-06-01,0,VALID_NON_EVENT
6,460383033365,2020-05-01,0,VALID_NON_EVENT
7,504708265545,2020-08-01,0,VALID_NON_EVENT
8,516964445282,2021-01-01,0,VALID_NON_EVENT
9,534283903679,2019-10-01,0,VALID_NON_EVENT


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

DIRECT TARGET LOGIC AUDIT


,loan_id,reporting_date,target_6m,target_status,status_t1,status_t2,status_t3,status_t4,status_t5,status_t6
0,269907063775,2018-04-01,0,VALID_NON_EVENT,00,00,00,00,00,00
1,227251848219,2018-07-01,0,VALID_NON_EVENT,00,00,00,00,00,00
2,215766289994,2018-08-01,0,VALID_NON_EVENT,00,00,00,00,00,00
3,534283903679,2019-10-01,0,VALID_NON_EVENT,00,00,00,00,00,00
4,460383033365,2020-05-01,0,VALID_NON_EVENT,00,00,00,00,00,00
5,504708265545,2020-08-01,0,VALID_NON_EVENT,00,00,00,00,00,00
6,516964445282,2021-01-01,0,VALID_NON_EVENT,00,00,00,00,00,00
7,252500752450,2021-11-01,0,VALID_NON_EVENT,00,00,00,00,00,00
8,171644560152,2022-08-01,0,VALID_NON_EVENT,00,00,00,00,00,00
9,450649322097,2025-06-01,0,VALID_NON_EVENT,00,00,00,00,00,00


# 6A. Direct Audit of Positive Early-Warning Events

### Objective
This section independently audits observations assigned `target_6m = 1`.

### Why this is necessary
The previous audit successfully confirmed that sampled non-events have six consecutive future months with status `00`.

A separate audit is required for positive events to verify that every sampled `target_6m = 1` observation has at least one future month with a delinquency status of `01` or greater.

This provides direct row-level evidence that the positive target was constructed according to the research definition.

No observations are modified.

In [9]:
# ================================================================
# DIRECT EVENT TARGET AUDIT
# ================================================================

# Select 10 deterministic event observations.
# ORDER BY hash() gives a reproducible sample without relying on
# USING SAMPLE behavior.

event_sample = target_con.execute("""
SELECT
    loan_id,
    reporting_date,
    target_6m,
    target_status
FROM early_warning_target
WHERE target_6m = 1
ORDER BY hash(loan_id, reporting_date)
LIMIT 10
""").fetchdf()

print("=" * 70)
print("SELECTED EVENT AUDIT SAMPLE")
print("=" * 70)

display(event_sample)

# ---------------------------------------------------------------
# Create a small VALUES table containing only these 10 events.
# ---------------------------------------------------------------

event_values = []

for _, row in event_sample.iterrows():

    event_values.append(
        f"""
        (
            {int(row["loan_id"])},
            DATE '{pd.Timestamp(row["reporting_date"]).strftime("%Y-%m-%d")}',
            {int(row["target_6m"])},
            '{row["target_status"]}'
        )
        """
    )

event_values_sql = ",\n".join(event_values)

# ---------------------------------------------------------------
# Retrieve the six actual future statuses.
# ---------------------------------------------------------------

event_audit_query = f"""
WITH sample AS (

    SELECT *
    FROM (
        VALUES
        {event_values_sql}
    ) AS v(
        loan_id,
        reporting_date,
        target_6m,
        target_status
    )

),

source_data AS (

    SELECT
        loan_id,

        STRPTIME(
            monthly_reporting_period,
            '%m%Y'
        ) AS reporting_date,

        current_delinquency_status

    FROM read_csv_auto(
        '{RAW_GLOB}',
        header = true,
        compression = 'gzip',
        union_by_name = true
    )

)

SELECT
    s.loan_id,
    s.reporting_date,
    s.target_6m,
    s.target_status,

    f1.current_delinquency_status AS status_t1,
    f2.current_delinquency_status AS status_t2,
    f3.current_delinquency_status AS status_t3,
    f4.current_delinquency_status AS status_t4,
    f5.current_delinquency_status AS status_t5,
    f6.current_delinquency_status AS status_t6

FROM sample s

LEFT JOIN source_data f1
    ON s.loan_id = f1.loan_id
   AND f1.reporting_date =
       s.reporting_date + INTERVAL '1 month'

LEFT JOIN source_data f2
    ON s.loan_id = f2.loan_id
   AND f2.reporting_date =
       s.reporting_date + INTERVAL '2 months'

LEFT JOIN source_data f3
    ON s.loan_id = f3.loan_id
   AND f3.reporting_date =
       s.reporting_date + INTERVAL '3 months'

LEFT JOIN source_data f4
    ON s.loan_id = f4.loan_id
   AND f4.reporting_date =
       s.reporting_date + INTERVAL '4 months'

LEFT JOIN source_data f5
    ON s.loan_id = f5.loan_id
   AND f5.reporting_date =
       s.reporting_date + INTERVAL '5 months'

LEFT JOIN source_data f6
    ON s.loan_id = f6.loan_id
   AND f6.reporting_date =
       s.reporting_date + INTERVAL '6 months'

ORDER BY
    s.reporting_date,
    s.loan_id
"""

event_audit_df = target_con.execute(
    event_audit_query
).fetchdf()

print("=" * 70)
print("DIRECT POSITIVE-EVENT TARGET AUDIT")
print("=" * 70)

display(event_audit_df)

SELECTED EVENT AUDIT SAMPLE


,loan_id,reporting_date,target_6m,target_status
0,527073029411,2018-02-01,1,VALID_EVENT
1,256697543094,2022-10-01,1,VALID_EVENT
2,671286197757,2020-02-01,1,VALID_EVENT
3,475979314461,2025-03-01,1,VALID_EVENT
4,865363760170,2021-07-01,1,VALID_EVENT
5,414106494625,2020-09-01,1,VALID_EVENT
6,443959547977,2020-09-01,1,VALID_EVENT
7,785221429437,2022-10-01,1,VALID_EVENT
8,746657459754,2018-10-01,1,VALID_EVENT
9,631268189916,2019-11-01,1,VALID_EVENT


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

DIRECT POSITIVE-EVENT TARGET AUDIT


,loan_id,reporting_date,target_6m,target_status,status_t1,status_t2,status_t3,status_t4,status_t5,status_t6
0,527073029411,2018-02-01,1,VALID_EVENT,00,00,00,00,01,00
1,746657459754,2018-10-01,1,VALID_EVENT,00,00,00,00,01,01
2,631268189916,2019-11-01,1,VALID_EVENT,00,00,00,00,01,02
3,671286197757,2020-02-01,1,VALID_EVENT,00,00,00,01,02,03
4,414106494625,2020-09-01,1,VALID_EVENT,00,00,00,01,00,00
5,443959547977,2020-09-01,1,VALID_EVENT,00,00,00,00,00,01
6,865363760170,2021-07-01,1,VALID_EVENT,00,01,01,02,03,03
7,256697543094,2022-10-01,1,VALID_EVENT,01,00,00,00,00,00
8,785221429437,2022-10-01,1,VALID_EVENT,00,00,00,01,00,00
9,475979314461,2025-03-01,1,VALID_EVENT,00,00,00,00,01,01


# 7. Save the Validated Early-Warning Target

### Objective
This section saves the validated six-month early-warning target as a permanent processed dataset.

### Why this is necessary
The target has now passed structural, temporal, and row-level validation. It should therefore be stored as a reproducible intermediate dataset before feature engineering and model development.

Only observations with a valid target (`0` or `1`) are retained in the modelling target file.

Observations excluded because of:
- missing future months; or
- unknown future delinquency status

are not included in the final modelling target.

### Important
The previously generated early-warning target file is not overwritten because it was constructed using an earlier methodology that did not explicitly enforce calendar continuity.

The corrected target is saved under a new filename.

In [10]:
# ================================================================
# SAVE VALIDATED TARGET
# ================================================================

TARGET_OUTPUT = (
    PROCESSED_DIR /
    "fannie_2018Q1_early_warning_target_corrected.parquet"
)

# Remove an existing corrected file only if it exists,
# so the output is reproducible when this cell is rerun.
if TARGET_OUTPUT.exists():
    TARGET_OUTPUT.unlink()

target_con.execute(f"""
COPY (
    SELECT
        loan_id,
        reporting_date,
        target_6m
    FROM early_warning_target
    WHERE target_6m IS NOT NULL
    ORDER BY
        reporting_date,
        loan_id
)
TO '{TARGET_OUTPUT}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
)
""")

# ---------------------------------------------------------------
# Verify the saved file
# ---------------------------------------------------------------

saved_target_check = target_con.execute(f"""
SELECT
    COUNT(*) AS observations,
    COUNT(DISTINCT loan_id) AS unique_loans,
    MIN(reporting_date) AS earliest_date,
    MAX(reporting_date) AS latest_date,
    COUNT(*) FILTER (
        WHERE target_6m = 1
    ) AS events,
    COUNT(*) FILTER (
        WHERE target_6m = 0
    ) AS non_events
FROM read_parquet('{TARGET_OUTPUT}')
""").fetchdf()

print("=" * 70)
print("VALIDATED TARGET SAVED")
print("=" * 70)

print(f"Output file: {TARGET_OUTPUT}")

display(saved_target_check)

print("\nFile size (MB):")
print(round(TARGET_OUTPUT.stat().st_size / (1024 ** 2), 2))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

VALIDATED TARGET SAVED
Output file: C:\Users\madhu\Downloads\MTP - Dynamic Credit Risk Early Warning\data\processed\fannie_2018Q1_early_warning_target_corrected.parquet


,observations,unique_loans,earliest_date,latest_date,events,non_events
0,20155269,450106,2018-01-01,2025-09-01,646957,19508312



File size (MB):
66.83
